# LangChain LLM Reference

Developer-facing statements defined in `langchain_core.language_models.llms`.


# `create_base_retry_decorator`

Creates a Tenacity retry decorator for selected exception types.

## Syntax

```python
create_base_retry_decorator(
    error_types: list[type[BaseException]], # Exception types that trigger retry
    max_retries: int = 1, # Maximum configured retry attempts
    run_manager: AsyncCallbackManagerForLLMRun
    | CallbackManagerForLLMRun
    | None = None, # Optional callback manager notified before retry
) -> Callable[[Any], Any] # Return the retry decorator
```

Uses exponential waiting between 4 and 10 seconds.

---

# `get_prompts`

Separates cached prompts from prompts that still require synchronous generation.

## Syntax

```python
get_prompts(
    params: dict[str, Any], # LLM parameters used to create the cache key
    prompts: list[str], # Prompts checked against the cache
    cache: BaseCache | bool | None = None, # Cache instance or cache setting
) -> tuple[
    dict[int, list[Generation]], # Cached generations indexed by prompt position
    str, # Serialized LLM cache key
    list[int], # Indexes of uncached prompts
    list[str], # Uncached prompts
]
```

Raises `ValueError` when `cache=True` and no global cache is configured.

---

# `aget_prompts`

Asynchronously separates cached prompts from prompts that still require generation.

## Syntax

```python
async aget_prompts(
    params: dict[str, Any], # LLM parameters used to create the cache key
    prompts: list[str], # Prompts checked against the cache
    cache: BaseCache | bool | None = None, # Cache instance or cache setting
) -> tuple[
    dict[int, list[Generation]], # Cached generations indexed by prompt position
    str, # Serialized LLM cache key
    list[int], # Indexes of uncached prompts
    list[str], # Uncached prompts
]
```

Raises `ValueError` when `cache=True` and no global cache is configured.

---

# `update_cache`

Stores newly generated results in the synchronous cache.

## Syntax

```python
update_cache(
    cache: BaseCache | bool | None, # Cache instance or cache setting
    existing_prompts: dict[int, list[Generation]], # Existing results indexed by prompt position
    llm_string: str, # Serialized LLM cache key
    missing_prompt_idxs: list[int], # Positions of newly generated prompts
    new_results: LLMResult, # Newly generated results
    prompts: list[str], # Complete prompt list
) -> dict[str, Any] | None # Return provider-specific LLM output
```

Raises `ValueError` when `cache=True` and no global cache is configured.

---

# `aupdate_cache`

Asynchronously stores newly generated results in the cache.

## Syntax

```python
async aupdate_cache(
    cache: BaseCache | bool | None, # Cache instance or cache setting
    existing_prompts: dict[int, list[Generation]], # Existing results indexed by prompt position
    llm_string: str, # Serialized LLM cache key
    missing_prompt_idxs: list[int], # Positions of newly generated prompts
    new_results: LLMResult, # Newly generated results
    prompts: list[str], # Complete prompt list
) -> dict[str, Any] | None # Return provider-specific LLM output
```

Raises `ValueError` when `cache=True` and no global cache is configured.


In [ ]:
import asyncio # Import asyncio for the asynchronous cache example
from typing import Any, Callable # Import types used in annotations

from langchain_core.caches import InMemoryCache # Import LangChain's in-memory LLM cache
from langchain_core.language_models.llms import ( # Import the LLM helper functions
    aget_prompts, # Import the asynchronous cache lookup function
    aupdate_cache, # Import the asynchronous cache update function
    create_base_retry_decorator, # Import the retry-decorator factory
    get_prompts, # Import the synchronous cache lookup function
    update_cache, # Import the synchronous cache update function
) # Finish importing the helper functions
from langchain_core.outputs import Generation, LLMResult # Import generation result types


attempts: dict[str, int] = {"count": 0} # Store the retry attempt count

retry_decorator: Callable[[Any], Any] = create_base_retry_decorator( # Create a retry decorator
    error_types=[ValueError], # Retry only when ValueError is raised
    max_retries=2, # Allow at most two total attempts
) # Finish creating the retry decorator


@retry_decorator # Apply the retry behaviour to the function
def unstable_operation(number: int) -> int: # Define an operation that fails temporarily
    attempts["count"] += 1 # Increase the attempt count

    print("Retry attempt:", attempts["count"]) # Display the current attempt

    if attempts["count"] == 1: # Fail during the first attempt
        raise ValueError("Temporary model failure") # Raise a retryable exception

    return number * 2 # Return the successful result


def synchronous_cache_example() -> None: # Demonstrate get_prompts and update_cache
    cache: InMemoryCache = InMemoryCache() # Create a synchronous in-memory cache

    params: dict[str, Any] = { # Define model parameters used in the cache key
        "model": "demo-llm", # Store the model name
        "temperature": 0, # Store the temperature
    } # Finish defining model parameters

    prompts: list[str] = [ # Define the complete prompt list
        "Explain Python", # Add the first prompt
        "Explain SQL", # Add the second prompt
    ] # Finish defining the prompts

    _, llm_string, _, _ = get_prompts( # Obtain the serialized LLM cache key
        params=params, # Supply the model parameters
        prompts=prompts, # Supply the prompts
        cache=cache, # Supply the in-memory cache
    ) # Finish the initial cache lookup

    cache.update( # Preload one cached generation
        prompts[0], # Use the first prompt as the cache key
        llm_string, # Use the generated LLM configuration key
        [Generation(text="Cached response about Python")], # Store one cached generation
    ) # Finish preloading the cache

    existing_prompts, llm_string, missing_indexes, missing_prompts = get_prompts( # Separate cached and missing prompts
        params=params, # Supply the model parameters
        prompts=prompts, # Supply the complete prompt list
        cache=cache, # Supply the cache
    ) # Finish checking the cache

    print("Synchronous cached indexes:", list(existing_prompts)) # Display cached prompt positions

    print("Synchronous missing indexes:", missing_indexes) # Display uncached prompt positions

    print("Synchronous missing prompts:", missing_prompts) # Display prompts requiring generation

    new_results: LLMResult = LLMResult( # Create results for only the missing prompts
        generations=[ # Define one generation list for each missing prompt
            [Generation(text=f"Generated response for: {prompt}")] # Create one generated response
            for prompt in missing_prompts # Process every missing prompt
        ], # Finish defining the generations
        llm_output={"provider": "demo-sync"}, # Add provider-specific output
    ) # Finish creating the LLM result

    llm_output: dict[str, Any] | None = update_cache( # Merge results and update the cache
        cache=cache, # Supply the cache
        existing_prompts=existing_prompts, # Supply already cached generations
        llm_string=llm_string, # Supply the serialized model key
        missing_prompt_idxs=missing_indexes, # Supply missing prompt positions
        new_results=new_results, # Supply newly generated results
        prompts=prompts, # Supply the complete prompt list
    ) # Finish updating the cache

    ordered_results: list[str] = [ # Restore results to the original prompt order
        existing_prompts[index][0].text # Read the first generation for the current prompt
        for index in range(len(prompts)) # Process every original prompt position
    ] # Finish creating the ordered result list

    print("Synchronous results:", ordered_results) # Display cached and generated results together

    print("Synchronous LLM output:", llm_output) # Display provider-specific output

    return # Finish the synchronous example


async def asynchronous_cache_example() -> None: # Demonstrate aget_prompts and aupdate_cache
    cache: InMemoryCache = InMemoryCache() # Create an asynchronous-compatible cache

    params: dict[str, Any] = { # Define model parameters used in the cache key
        "model": "demo-async-llm", # Store the model name
        "temperature": 0, # Store the temperature
    } # Finish defining model parameters

    prompts: list[str] = [ # Define the asynchronous prompt list
        "Explain LangChain", # Add the first prompt
        "Explain Runnables", # Add the second prompt
    ] # Finish defining the prompts

    _, llm_string, _, _ = await aget_prompts( # Obtain the asynchronous LLM cache key
        params=params, # Supply the model parameters
        prompts=prompts, # Supply the prompts
        cache=cache, # Supply the cache
    ) # Finish the initial asynchronous lookup

    await cache.aupdate( # Preload one cached asynchronous generation
        prompts[0], # Use the first prompt as the cache key
        llm_string, # Use the generated LLM configuration key
        [Generation(text="Cached response about LangChain")], # Store one generation
    ) # Finish preloading the asynchronous cache

    existing_prompts, llm_string, missing_indexes, missing_prompts = await aget_prompts( # Separate cached and missing prompts
        params=params, # Supply the model parameters
        prompts=prompts, # Supply the complete prompt list
        cache=cache, # Supply the cache
    ) # Finish checking the asynchronous cache

    print("Asynchronous cached indexes:", list(existing_prompts)) # Display cached prompt positions

    print("Asynchronous missing indexes:", missing_indexes) # Display uncached prompt positions

    new_results: LLMResult = LLMResult( # Create results for the missing prompts
        generations=[ # Define one generation list for each missing prompt
            [Generation(text=f"Generated response for: {prompt}")] # Create one generated response
            for prompt in missing_prompts # Process every missing prompt
        ], # Finish defining generations
        llm_output={"provider": "demo-async"}, # Add provider-specific output
    ) # Finish creating the asynchronous LLM result

    llm_output: dict[str, Any] | None = await aupdate_cache( # Merge results and asynchronously update the cache
        cache=cache, # Supply the cache
        existing_prompts=existing_prompts, # Supply already cached generations
        llm_string=llm_string, # Supply the serialized model key
        missing_prompt_idxs=missing_indexes, # Supply missing prompt positions
        new_results=new_results, # Supply newly generated results
        prompts=prompts, # Supply the complete prompt list
    ) # Finish asynchronously updating the cache

    ordered_results: list[str] = [ # Restore the original prompt order
        existing_prompts[index][0].text # Read the first generation for the prompt
        for index in range(len(prompts)) # Process every prompt position
    ] # Finish creating the ordered results

    print("Asynchronous results:", ordered_results) # Display cached and generated results

    print("Asynchronous LLM output:", llm_output) # Display provider-specific output

    return # Finish the asynchronous example


retry_result: int = unstable_operation(10) # Execute the function with retry behaviour

print("Retry result:", retry_result) # Display the successful retry result

synchronous_cache_example() # Run the synchronous cache example

asyncio.run(asynchronous_cache_example()) # Run the asynchronous cache example

# `BaseLLM: BaseLanguageModel[str], ABC`

`BaseLLM` is the abstract base class for traditional text-completion models.

Concrete subclasses must implement `_generate()` and `_llm_type`.

## Overridden Properties and Methods

### `OutputType`

Returns `str` as the Runnable output type.

### `invoke`

Synchronously accepts a string, message sequence, or `PromptValue` and returns one generated string.

### `ainvoke`

Asynchronously accepts a string, message sequence, or `PromptValue` and returns one generated string.

### `batch`

Generates strings for multiple inputs.

Uses `max_concurrency` from `RunnableConfig` when supplied.

### `abatch`

Asynchronously generates strings for multiple inputs.

Uses `max_concurrency` from `RunnableConfig` when supplied.

### `stream`

Synchronously yields generated text chunks.

Falls back to `invoke()` when `_stream()` is not implemented.

### `astream`

Asynchronously yields generated text chunks.

Falls back to `ainvoke()` when neither `_stream()` nor `_astream()` is implemented.

### `generate_prompt`

Converts each `PromptValue` to a string and delegates to `generate()`.

### `agenerate_prompt`

Asynchronously converts each `PromptValue` to a string and delegates to `agenerate()`.

## Methods

### `generate`

Generates results for multiple string prompts.

```python
generate(
    prompts: list[str], # Prompts supplied to the model
    stop: list[str] | None = None, # Optional stop sequences
    callbacks: Callbacks | list[Callbacks] | None = None, # Shared or per-prompt callbacks
    *,
    tags: list[str] | list[list[str]] | None = None, # Shared or per-prompt tags
    metadata: dict[str, Any]
    | list[dict[str, Any]]
    | None = None, # Shared or per-prompt metadata
    run_name: str | list[str] | None = None, # Shared or per-prompt run names
    run_id: UUID
    | list[UUID | None]
    | None = None, # Shared or per-prompt run identifiers
    **kwargs: Any, # Provider-specific generation arguments
) -> LLMResult # Return generations and provider-specific output
```

Raises `ValueError` when `prompts` is not a list or per-prompt configuration lengths do not match.

### `agenerate`

Asynchronously generates results for multiple string prompts.

```python
async agenerate(
    prompts: list[str], # Prompts supplied to the model
    stop: list[str] | None = None, # Optional stop sequences
    callbacks: Callbacks | list[Callbacks] | None = None, # Shared or per-prompt callbacks
    *,
    tags: list[str] | list[list[str]] | None = None, # Shared or per-prompt tags
    metadata: dict[str, Any]
    | list[dict[str, Any]]
    | None = None, # Shared or per-prompt metadata
    run_name: str | list[str] | None = None, # Shared or per-prompt run names
    run_id: UUID
    | list[UUID | None]
    | None = None, # Shared or per-prompt run identifiers
    **kwargs: Any, # Provider-specific generation arguments
) -> LLMResult # Return generations and provider-specific output
```

Raises `ValueError` when per-prompt configuration lengths do not match.

### `dict`

Deprecated since `1.4.2` and scheduled for removal in `2.0.0`.

Use `asdict()` instead.

### `asdict`

Returns the identifying model parameters together with `_type`.

```python
asdict(
    self, # Current LLM instance
) -> dict[str, Any] # Return the model dictionary
```

### `save`

Saves the identifying model parameters as JSON or YAML.

```python
save(
    self, # Current LLM instance
    file_path: Path | str, # Destination ending in .json, .yaml, or .yml
) -> None # Save the model configuration
```

Raises `ValueError` when the destination is not a JSON or YAML file.

## Required Subclass Hooks

### `_generate`

Generates an `LLMResult` for multiple prompts.

```python
_generate(
    self, # Current LLM instance
    prompts: list[str], # Prompts supplied to the model
    stop: list[str] | None = None, # Optional stop sequences
    run_manager: CallbackManagerForLLMRun | None = None, # Synchronous callback manager
    **kwargs: Any, # Provider-specific generation arguments
) -> LLMResult # Return the generated results
```

### `_llm_type`

Returns the string identifying the LLM type.

```python
@property
def _llm_type(
    self, # Current LLM instance
) -> str # Return the model type identifier
```

## Optional Subclass Hooks

### `_agenerate`

Provides native asynchronous batched generation.

The default implementation runs `_generate()` in an executor.

### `_stream`

Provides synchronous generation streaming.

The default implementation raises `NotImplementedError`.

### `_astream`

Provides native asynchronous generation streaming.

The default implementation adapts `_stream()` through an executor.

## Behaviour

- Accepts strings, `PromptValue` objects, and message sequences.
- Supports synchronous, asynchronous, batch, cache, callback, and streaming execution.
- Returns plain strings from Runnable invocation.
- Returns complete `LLMResult` objects from `generate()` and `agenerate()`.

---

# `LLM: BaseLLM`

`LLM` is an abstract convenience base class for models that generate one string per prompt.

Concrete subclasses must implement `_call()`, `_llm_type`, and identifying parameters.

## Required Subclass Hook

### `_call`

Generates one string from one prompt.

```python
_call(
    self, # Current LLM instance
    prompt: str, # Prompt supplied to the model
    stop: list[str] | None = None, # Optional stop sequences
    run_manager: CallbackManagerForLLMRun | None = None, # Synchronous callback manager
    **kwargs: Any, # Provider-specific generation arguments
) -> str # Return generated text without the original prompt
```

## Optional Subclass Hook

### `_acall`

Provides native asynchronous generation for one prompt.

```python
async _acall(
    self, # Current LLM instance
    prompt: str, # Prompt supplied to the model
    stop: list[str] | None = None, # Optional stop sequences
    run_manager: AsyncCallbackManagerForLLMRun | None = None, # Asynchronous callback manager
    **kwargs: Any, # Provider-specific generation arguments
) -> str # Return generated text without the original prompt
```

The default implementation runs `_call()` in an executor.

## Implemented Methods

### `_generate`

Calls `_call()` for each prompt and wraps the strings in `Generation` and `LLMResult` objects.

### `_agenerate`

Calls `_acall()` for each prompt and wraps the strings in `Generation` and `LLMResult` objects.